In [1]:
from google import genai
from google.genai import types
import os
import json
from IPython.display import display, Markdown

In [2]:
import qdrant_client
from qdrant_client import models
qdrant_client = qdrant_client.AsyncQdrantClient('http://localhost:6333', timeout=1000)
GEMINI_API_KEY = os.getenv('GEMINI_TOKEN')
GEMINI_MODEL = 'gemini-2.5-flash'
# Only run this block for Gemini Developer API
client = genai.Client(api_key=GEMINI_API_KEY)

In [3]:
QUERY = "Quando è nato Dante Alighieri?"

# Definiamo la fase di query parsing

## Il primo step è la query rewriting

In [4]:
def ask_gemini_to_rewrite_the_query_in_hyve_setup(query: str):
    system_message = """You are an AI language model assistant. Your task is to generate hypothetical passages in less than 300 words in Italian that can answer a given question. If you don't know the answer generate a plausible passage."""
    prompt_template = """Please write a passage to answer the question.
    query: {{query}}
    \n\n"""
    prompt = prompt_template.replace('{{query}}', query)
    parts = [
        types.Part.from_text(text=prompt),
    ]
    content_list = [
        types.Content(
            role='user',
            parts=parts
        )
    ]
    result = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=content_list,
            config=types.GenerateContentConfig(
                system_instruction=system_message,
                temperature=0.3)
    )
    return result.text

    

In [5]:
rewrited_query = ask_gemini_to_rewrite_the_query_in_hyve_setup(QUERY)

In [6]:
rewrited_query

"La data esatta di nascita di Dante Alighieri non è conosciuta con certezza, ma gli studiosi concordano nel collocarla tra la fine di maggio e l'inizio di giugno del 1265.\n\nNon esistono documenti ufficiali o registri anagrafici dell'epoca che attestino il giorno preciso. Tuttavia, questa stima si basa su diverse fonti indirette, tra cui riferimenti astrologici presenti nelle sue stesse opere, in particolare nella *Divina Commedia*. Ad esempio, nel *Paradiso*, Dante allude al fatto di essere nato sotto il segno dei Gemelli, che corrisponde appunto al periodo tra la fine di maggio e la fine di giugno.\n\nInoltre, altre testimonianze e tradizioni biografiche antiche supportano questa collocazione temporale. Pertanto, sebbene non si possa indicare un giorno specifico, il periodo tra la fine di maggio e l'inizio di giugno del 1265 è universalmente accettato come il momento della nascita del Sommo Poeta."

## Il secondo step è l'estrazione delle entità

In [7]:
from pydantic import BaseModel
from typing import List
class Entities(BaseModel):
    entities: List[str]

In [8]:
def ask_gemini_to_extract_entities(query: str):
    system_message = """    You are an helpful and skilled analyst, and you are very good in extracting entities from a given query. 
    You are presented with a text and you are asked to extract entities from it.
    ## GUIDELINES
    1. Extract only Organization names, Locations and person names
    3. Answer in Italian"""
    prompt_template = """Extract entities from the given query: \n\n{{query}}"""
    prompt = prompt_template.replace('{{query}}', query)
    parts = [
        types.Part.from_text(text=prompt),
    ]
    content_list = [
        types.Content(
            role='user',
            parts=parts
        )
    ]
    result = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=content_list,
            config=types.GenerateContentConfig(
                system_instruction=system_message,
                temperature=0.3,
                response_mime_type='application/json',
            response_schema=Entities)
    )
    return result.parsed

In [9]:
entities = ask_gemini_to_extract_entities(QUERY).entities

In [10]:
entities

['Dante Alighieri']

In [ ]:
class TimeRange(BaseModel):
    start_date: str = Field(description='IN YYYY-MM-DD format')
    end_date: str

## Ci possono essere anche altri step come l'estrazione dei timerange o l'estrazione dei filename ma noi non li consideriamo

# Definiamo la fase di retrieval

## Retrieval senza entità e senza Hyve

In [11]:
from fastembed import TextEmbedding, SparseTextEmbedding, LateInteractionTextEmbedding

dense_embedding_model = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2", cache_dir = './fastembed/')
bm25_embedding_model = SparseTextEmbedding("Qdrant/bm25", cache_dir = './fastembed/')
late_interaction_embedding_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")

In [12]:
dense_embeddings = list(dense_embedding_model.embed(QUERY))[0]
bm25_embeddings = list(bm25_embedding_model.embed(QUERY))[0]
late_interaction_embeddings = list(late_interaction_embedding_model.embed(QUERY))[0]

In [14]:
prefetch = [
        models.Prefetch(
            query=dense_embeddings,
            using="dense",
            limit=20,
        ),
        models.Prefetch(
            query=models.SparseVector(**bm25_embeddings.as_object()),
            using="bm25",
            limit=20,
        ),
    ]

results = await qdrant_client.query_points(
         "pdf",
        prefetch=prefetch,
        query=late_interaction_embeddings,
        using="colbert",
        with_payload=True,
        limit=10,
)

In [15]:
results_no_entity = [res.payload['document'] for res in  results.points]

In [18]:
results_no_entity[0]

'Ecco un abstract eccellente basato sugli insight forniti:\n\n**Dante Alighieri: Architetto della Cultura Occidentale e Faro dell\'Umanità**\n\nDante Alighieri si erge come figura centrale della cultura occidentale, non solo apice della letteratura medievale ma cerniera ontologica tra teocentrismo e nascente umanesimo. La sua opera, una "cattedrale di parole", integra armoniosamente filosofia aristotelica, teologia tomista, passione civile e sperimentazione linguistica, configurandosi come un compendio universale del sapere e dell\'esperienza umana del suo tempo.\n\nNato a Firenze nel 1265 in un contesto di profonde tensioni politiche, Dante fu attivamente coinvolto nella vita comunale, culminata nella carica di Priore. La sua formazione poliedrica, influenzata da Brunetto Latini, lo preparò all\'impegno civile. Tuttavia, l\'ingiusta condanna all\'esilio nel 1302, per ragioni politiche, segnò una frattura biografica che trasformò il trauma personale in una missione profetica universale

## Retrieval con entità

In [20]:
def create_should_clause(entities: List[str]):
    should = []
    for entity in entities:
        should.append(models.FieldCondition(key='metadata.entities[]', match=models.MatchText(text=entity)))
    return should
    

prefetch = [
        models.Prefetch(
            query=dense_embeddings,
            using="dense",
            limit=20,
            filter= models.Filter(should=create_should_clause(entities))
        ),
        models.Prefetch(
            query=models.SparseVector(**bm25_embeddings.as_object()),
            using="bm25",
            limit=20,
            filter= models.Filter(should=create_should_clause(entities))
        ),
    ]

results = await qdrant_client.query_points(
         "pdf",
        prefetch=prefetch,
        query=late_interaction_embeddings,
        using="colbert",
        with_payload=True,
        limit=10,
)

In [21]:
results_entity = [res.payload['document'] for res in  results.points]

In [27]:
results_entity[0]

'Ecco un abstract eccellente basato sugli insight forniti:\n\n**Dante Alighieri: Architetto della Cultura Occidentale e Faro dell\'Umanità**\n\nDante Alighieri si erge come figura centrale della cultura occidentale, non solo apice della letteratura medievale ma cerniera ontologica tra teocentrismo e nascente umanesimo. La sua opera, una "cattedrale di parole", integra armoniosamente filosofia aristotelica, teologia tomista, passione civile e sperimentazione linguistica, configurandosi come un compendio universale del sapere e dell\'esperienza umana del suo tempo.\n\nNato a Firenze nel 1265 in un contesto di profonde tensioni politiche, Dante fu attivamente coinvolto nella vita comunale, culminata nella carica di Priore. La sua formazione poliedrica, influenzata da Brunetto Latini, lo preparò all\'impegno civile. Tuttavia, l\'ingiusta condanna all\'esilio nel 1302, per ragioni politiche, segnò una frattura biografica che trasformò il trauma personale in una missione profetica universale

## Retrieval con hyve

In [23]:
dense_embeddings = list(dense_embedding_model.embed(rewrited_query))[0]
bm25_embeddings = list(bm25_embedding_model.embed(rewrited_query))[0]
late_interaction_embeddings = list(late_interaction_embedding_model.embed(rewrited_query))[0]

In [24]:
prefetch = [
        models.Prefetch(
            query=dense_embeddings,
            using="dense",
            limit=20,
        ),
        models.Prefetch(
            query=models.SparseVector(**bm25_embeddings.as_object()),
            using="bm25",
            limit=20,
        ),
    ]

results = await qdrant_client.query_points(
         "pdf",
        prefetch=prefetch,
        query=late_interaction_embeddings,
        using="colbert",
        with_payload=True,
        limit=10,
)

In [25]:
results_hyve = [res.payload['document'] for res in  results.points]

In [26]:
results_hyve[0]

'Dante Alighieri nasce a Firenze tra il 21 maggio e il 21 giugno del 1265, in un clima di profonda tensione sociale e politica. <sup>1</sup> La sua famiglia apparteneva alla piccola nobiltà guelfa, una classe sociale che, pur non disponendo di ingenti risorse economiche, partecipava attivamente alla vita militare e civile del Comune. <sup>2</sup> Le radici di Dante affondano in una Firenze in piena espansione, dove la lotta tra Guelfi (sostenitori del Papato) e Ghibellini (sostenitori dell\'Impero) non era solo un conflitto ideologico, ma una disputa per l\'egemonia commerciale e politica. 4  \nL\'educazione di Dante fu poliedrica e rifletteva lo sperimentalismo tipico dei centri urbani toscani del XIII secolo. Sebbene le fonti sulla sua infanzia siano scarse, è accertato che ricevette i primi rudimenti di grammatica e latino a Firenze, approfondendo poi gli studi filosofici presso le scuole degli ordini mendicanti, francescani e domenicani. <sup>2</sup> Un ruolo cruciale nella sua for

# Definiamo la fase di post retrieval: reranking

In [28]:
results = set(results_entity + results_hyve + results_no_entity)

In [29]:
len(results)

15

In [31]:
from transformers import AutoModel

model = AutoModel.from_pretrained(
    'jinaai/jina-reranker-v3',
    dtype="auto",
    trust_remote_code=True,
)
model.eval()

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   4%|3         | 41.9M/1.19G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/202 [00:00<?, ?B/s]

JinaForRanking(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layerno

In [32]:
results = model.rerank(QUERY, list(results))

# Results are sorted by relevance score (highest first)
for result in results:
    print(f"Score: {result['relevance_score']:.4f}")
    print(f"Document: {result['document'][:100]}...")
    print()

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

Score: 0.2975
Document: Dante Alighieri nacque a Firenze tra il 21 maggio e il 21 giugno del 1265, in un clima di profonda t...

Score: 0.2532
Document: Dante Alighieri nasce a Firenze tra il 21 maggio e il 21 giugno del 1265, in un clima di profonda te...

Score: 0.1874
Document: - 1. Alighieri, Dante Enciclopedia Treccani, accesso eseguito il giorno marzo 27, 2026, [https://www...

Score: 0.0847
Document: L'incontro con Beatrice Portinari, storicamente identificata con Bice di Folco Portinari, rappresent...

Score: 0.0143
Document: L'incontro con Beatrice Portinari, identificata storicamente con Bice di Folco Portinari, rappresent...

Score: -0.0191
Document: Il 27 gennaio 1302, mentre era a Roma come ambasciatore presso Bonifacio VIII, Dante Alighieri fu co...

Score: -0.0640
Document: Il pensiero politico di Dante Alighieri trova la sua espressione più compiuta e audace nel *De Monar...

Score: -0.0928
Document: Il pensiero politico di Dante trova la sua espressione più compiuta e 

In [112]:
results[6]['document']

'L\'esperienza maturata sul campo rimane il driver principale per la definizione della base salariale, sebbene la rapidità con cui un professionista acquisisce familiarità con i nuovi framework (come LangChain, PyTorch Lightning o Hugging Face) possa accelerare notevolmente gli scatti retributivi. 2  \nQuì c\'era una tabella che è stata sostituita dalla sua descrizione.\nDescrizione della tabella: Questa tabella presenta una panoramica delle retribuzioni e dei bonus stimati in base al livello di seniority. È strutturata in quattro colonne che forniscono informazioni dettagliate per ogni categoria di esperienza.  \nLa prima riga della tabella funge da intestazione e descrive il contenuto di ciascuna colonna. La prima colonna, intitolata "Livello di Seniority", indica il grado di esperienza professionale. La seconda colonna, "RAL Media Nazionale (€)", mostra la Retribuzione Annua Lorda media a livello nazionale espressa in euro. La terza colonna, "Range Minimo-Massimo (€)", specifica l\'

## Definiamo la fase di generazione

In [33]:
def ask_gemini_to_answer_query(query: str, documents: List[str]):
    system_message = """You are an excellent AI assitant that answer user queries given relevant documents. The documents are sorted by relevance (higher means more relevant).
    Rispondi solo con affermazioni fattuali basati sui documenti recuperati.
    Cita le tue fonti alla fine di ogni frase utilizzando [1], [2] ecc
    """
    prompt_template = """Answer the given query using the given context documents: \n\nQUERY: {{query}}. \n\nDOCUMENTS: {{documents}}"""
    prompt = prompt_template.replace('{{query}}', query).replace('{{documents}}', '\n\n'.join(documents))
    parts = [
        types.Part.from_text(text=prompt),
    ]
    content_list = [
        types.Content(
            role='user',
            parts=parts
        )
    ]
    result = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=content_list,
            config=types.GenerateContentConfig(
                system_instruction=system_message,
                temperature=0.3)
    )
    return result.text

In [34]:
answer = ask_gemini_to_answer_query(QUERY, [result['document'] for result in results[:5]])

In [36]:
results[0]

{'document': 'Dante Alighieri nacque a Firenze tra il 21 maggio e il 21 giugno del 1265, in un clima di profonda tensione sociale e politica.\n\nLa sua famiglia apparteneva alla piccola nobiltà guelfa, che partecipava attivamente alla vita militare e civile del Comune di Firenze.\n\nFirenze era in piena espansione, caratterizzata dalla lotta tra Guelfi (sostenitori del Papato) e Ghibellini (sostenitori dell\'Impero) per l\'egemonia commerciale e politica.\n\nL\'educazione di Dante fu poliedrica, ricevendo i primi rudimenti a Firenze e approfondendo gli studi filosofici presso le scuole degli ordini mendicanti (francescani e domenicani).\n\nBrunetto Latini svolse un ruolo cruciale nella formazione di Dante, insegnandogli la retorica e una concezione della cultura come strumento di impegno civile e politico.\n\nDante partecipò alla vita militare del Comune, combattendo nella battaglia di Campaldino del 1289 contro i ghibellini di Arezzo.\n\nNel 1295, si iscrisse all\'Arte dei Medici e Sp

In [35]:
display(Markdown(answer))

Dante Alighieri nacque a Firenze tra il 21 maggio e il 21 giugno del 1265 [1].

In [129]:
[result['document'] for result in results[:5]]

['Milano si conferma il polo gravitazionale per l\'intelligenza artificiale, ospitando la maggior parte delle multinazionali tech e delle startup ad alta capitalizzazione. Qui, la retribuzione media per un AI Engineer è superiore del 10,5% rispetto alla media nazionale. 4  \nQuì c\'era una tabella che è stata sostituita dalla sua descrizione.\nDescrizione della tabella: La tabella presentata è strutturata in tre colonne distinte.\nLa prima colonna è intitolata "Area Geografica" e indica le diverse località o regioni considerate.\nLa seconda colonna è intitolata "RAL Media AI Engineer (€)" e riporta la Retribuzione Annua Lorda media in euro per un Ingegnere AI.\nLa terza colonna è intitolata "Differenziale vs Media Italia" e mostra la differenza percentuale rispetto alla media italiana della RAL.  \nPassando alla descrizione di ogni riga:  \nLa prima riga della tabella è l\'intestazione e definisce il contenuto delle colonne: "Area Geografica", "RAL Media AI Engineer (€)" e "Differenzia